In [1]:
#FRAMEWORK ULTIMA VERSIONE UFFICIALE IMPLEMENTATO V2
#CLASSI GENERICHE DI TRASFORMAZIONE SYNTH DI UN DF CON XGBOOST CON PREDICT_PROBA 
#bug fix eliminato synth su colonne reali

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from typing import Dict, List, Optional
import os


class XGBoostSynthesizer:
    """
    Sintetizzatore tabellare incrementale/autoregressivo
    - Train su df reale
    - Sampling su df sintetico parziale
    - Colonna per colonna
    - Rumore regolabile
    """

    def __init__(
        self,
        random_state: int = 42,
        add_noise: bool = False,
        noise_factor: float = 0.05,
        n_estimators: int = 300,
        max_depth: int = 6
    ):
        self.random_state = random_state
        self.add_noise = add_noise          # Se True aggiunge rumore
        self.noise_factor = noise_factor    # Frazione std per rumore
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.label_encoders: Dict[str, LabelEncoder] = {}
        self.original_dtypes: Dict[str, str] = {}
        self.real_columns: List[str] = []
        self.df_real: Optional[pd.DataFrame] = None

        pd.options.mode.copy_on_write = True
        np.random.seed(self.random_state)

    # =====================================================
    # PUBLIC API
    # =====================================================
    def fit_sample(
        self,
        df: pd.DataFrame,
        sample_size: Optional[int] = None,
        column_order: Optional[List[str]] = None
    ) -> pd.DataFrame:

        if sample_size is None:
            sample_size = len(df)

        self.df_real = df.copy()
        self.real_columns = df.columns.tolist()
        self.original_dtypes = df.dtypes.to_dict()

        columns_data = self._generate_all_columns(
            df=df,
            sample_size=sample_size,
            column_order=column_order
        )

        df_synth = pd.DataFrame(columns_data)
        return self._restore_dtypes(df_synth)

    # =====================================================
    # CORE LOGIC
    # =====================================================
    def _generate_all_columns(
        self,
        df: pd.DataFrame,
        sample_size: int,
        column_order: Optional[List[str]]
    ) -> Dict[str, np.ndarray]:

        # ordine colonne se non specificato prima le numeriche poi le categoriali
        if column_order is None:
            num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
            cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
            synthesis_order = num_cols + cat_cols
        else:
            synthesis_order = [c for c in column_order if c in df.columns]

        print(f"🔄 Sintesi incrementale: {synthesis_order}")

        columns_data: Dict[str, np.ndarray] = {}

        # ---------- PRIMA COLONNA (BOOTSTRAP PURO)
        first_col = synthesis_order[0]
        columns_data[first_col] = df[first_col].sample(
            n=sample_size,
            replace=True,
            random_state=self.random_state
        ).values

        # ---------- COLONNE SUCCESSIVE (INCREMENTALE)
        for col in synthesis_order[1:]:
            print(f"🔄 Sintetizzando {col}")
            df_synth_partial = pd.DataFrame(columns_data)

            synth_col = self._synthesize_single_column(
                df_real=df,
                df_synth_partial=df_synth_partial,
                target_col=col
            )

            columns_data[col] = synth_col

        return columns_data

    def _synthesize_single_column(
        self,
        df_real: pd.DataFrame,
        df_synth_partial: pd.DataFrame,
        target_col: str
    ) -> np.ndarray:

        feature_cols = df_synth_partial.columns.tolist()

        # -----------------------------
        # TRAIN (REAL DATA)
        # -----------------------------
        X_train = self._prepare_features(df_real[feature_cols])
        y_train = df_real[target_col]

        # -----------------------------
        # SAMPLE (SYNTH DATA)
        # -----------------------------
        X_synth = self._prepare_features(df_synth_partial)
        X_synth = self._align_features(X_train, X_synth)

        is_categorical = (
            y_train.dtype in ["object", "category","str"]
        )

        # =================================================
        # CATEGORICA
        # =================================================
        if is_categorical:
            print(f"🔄 è categorica")
            y_enc = self._encode_target(y_train, target_col)

            model = xgb.XGBClassifier(
                n_estimators=self.n_estimators,
                max_depth=self.max_depth,
                learning_rate=0.05,
                subsample=0.7,
                colsample_bytree=0.7,
                eval_metric="mlogloss",
                random_state=self.random_state
            )

            model.fit(X_train, y_enc)
            
            #con predict_proba mantengo la variabilità naturale dei dati invece di usare sempre la classe più probabile.
            proba = model.predict_proba(X_synth)
            #print(proba)

            #itera per ogni riga usando array di proba con le p per ogni classe di ogni record          
            #sceglie casualmente ma la casualità è ponderata con le p 
            sampled = np.array([
                np.random.choice(len(p), p=p)
                for p in proba
            ])

            return self._decode_target(sampled, target_col)

        # =================================================
        # NUMERICA
        # =================================================
        else:
            print(f"🔄 è numerica")
            model = xgb.XGBRegressor(
                n_estimators=self.n_estimators,
                max_depth=self.max_depth,
                learning_rate=0.05,
                subsample=0.7,
                colsample_bytree=0.7,
                random_state=self.random_state
            )

            model.fit(X_train, y_train)

            preds = model.predict(X_synth)
            print(len(preds))
            if self.add_noise:
                noise_std = max(np.std(y_train) * self.noise_factor, 1e-6)
                noise = np.random.normal(0, noise_std, size=len(preds))
                preds = preds + noise

            return np.clip(preds, y_train.min(), y_train.max())

    # =====================================================
    # UTILS
    # =====================================================
    def _prepare_features(self, df: pd.DataFrame) -> pd.DataFrame:
        return pd.get_dummies(df, prefix_sep="_")

    def _align_features(
        self,
        X_train: pd.DataFrame,
        X_synth: pd.DataFrame
    ) -> pd.DataFrame:

        missing = set(X_train.columns) - set(X_synth.columns)
        for col in missing:
            X_synth[col] = 0

        return X_synth[X_train.columns]

    def _encode_target(self, y: pd.Series, col_name: str) -> np.ndarray:
        if col_name not in self.label_encoders:
            le = LabelEncoder()
            self.label_encoders[col_name] = le.fit(y)
        return self.label_encoders[col_name].transform(y)

    def _decode_target(self, y_enc: np.ndarray, col_name: str) -> np.ndarray:
        return self.label_encoders[col_name].inverse_transform(y_enc)

    def _restore_dtypes(self, df_synth: pd.DataFrame) -> pd.DataFrame:
        df_out = df_synth.copy()

        for col, dtype in self.original_dtypes.items():
            if col not in df_out.columns:
                continue

            try:
                dtype_str = str(dtype).lower()
                s = df_out[col]

                if "int" in dtype_str:
                    df_out[col] = s.fillna(0).round().astype("int64")
                elif "float" in dtype_str:
                    df_out[col] = s.astype("float64")
                elif "bool" in dtype_str:
                    df_out[col] = s.astype("boolean")
                elif "datetime" in dtype_str:
                    df_out[col] = pd.to_datetime(s, errors="coerce")
                else:
                    df_out[col] = s.astype("object")

            except Exception as e:
                print(f"⚠️ Errore dtype {col}: {e}")

        return df_out.reindex(columns=self.real_columns)


## Tipologia di ordine 
O1 ordine_random_1 casuale ma con ordine pensato in base alla formula

O2 ordine_random_2 casuale ma con ordine pensato all'ordine demografico

O3 ordine_random_3 casuale puro

O4 ordine_piramide_rov dalla più numerose categoriche alle meno

O5 ordine_piramide_inc dalle minori numerose categoriche alle più numerose

O6 ordine_mi ordine di importanza delle features dato da analisi MI (Mutual Information)


In [2]:
o0=['age', 'physical_activity', 'genetic_predisposition','municipality_residence', 'civil_status', 'gender','occupation',  'diagnosis']
o1=['municipality_residence','age', 'civil_status', 'gender','occupation','physical_activity', 'genetic_predisposition',  'diagnosis']
o2=['gender','age','physical_activity','civil_status','occupation','genetic_predisposition','municipality_residence','diagnosis']
o3=['municipality_residence','age','genetic_predisposition','physical_activity','civil_status','gender','occupation','diagnosis']
o4=['occupation','gender','civil_status','physical_activity','genetic_predisposition','age','municipality_residence','diagnosis']
o5=['physical_activity', 'genetic_predisposition', 'age', 'occupation', 'gender', 'municipality_residence', 'civil_status',  'diagnosis']

#scegliere un numero da 0 a 5 per stabilire ordinamento da far girare
ordine_scelto=0 

ordine_all=[o0,o1,o2,o3,o4,o5]
ordine_ottimale=ordine_all[ordine_scelto] #l'indice determina la scelta

print(ordine_ottimale)
print(ordine_scelto)


['age', 'physical_activity', 'genetic_predisposition', 'municipality_residence', 'civil_status', 'gender', 'occupation', 'diagnosis']
0


In [3]:
##I test verranno fatti con R=Y e PR=10 specifica Ricciato
np.random.seed(42)

R='Y' # da settare

if R != "N":
    PR=100 #da settare
else:
    PR=0


synthesizer_type="XGBoost"
nome_utente = 'spagnuol'

input_dir =  rf'c:\Users\{nome_utente}\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step1\Output'
output_dir = rf'c:\Users\{nome_utente}\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step2\Output'


input_file_name = os.path.join(
    input_dir,
    f"real_data_datasetM10_TH20_R{R}_PR{PR}_4CAT_MISS_X2.csv"
)

output_file_name = os.path.join(
    output_dir,
    f"synthetic_data_datasetM10_TH20_R{R}_PR{PR}_4CAT_MISS_X2_{synthesizer_type}_O{ordine_scelto}.csv"
)

print(input_file_name)
print(output_file_name)

df_real = pd.read_csv(input_file_name, sep=',')

columns_to_drop = ['id', 'score1', 'score2', 'bin1', 'bin2', 'raw_class', 'age_norm', 'activity_norm', 'predisposition_norm','birth_year','birth_month','birth_day',
                   'municipality_birth','birth_dayofyear']
df_real = df_real.drop(columns=columns_to_drop)


###VERSIONE AGGIORNATA CON COLONNE NUMERICHE O CATEGORICHE CUSTOMIZZABILI
#cols_num = ["diagnosis"]
cols_num = []
df_real[cols_num] = (
    df_real[cols_num]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
    .astype("int64")
)

# Trasformazione IN CATEGORICHE
cols_cat = ["physical_activity", "genetic_predisposition","occupation","age","diagnosis"] 
df_real[cols_cat] = (
    df_real[cols_cat]
    .astype(str)  # Converti tutto a stringa
    .replace(['nan', 'NaN', 'None', 'none', ''], 'missing')  # Standardizza missing
    .fillna('missing')  # Fill NaN
    .apply(lambda x: x.str.strip())  # Rimuovi spazi
)



c:\Users\spagnuol\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step1\Output\real_data_datasetM10_TH20_RY_PR100_4CAT_MISS_X2.csv
c:\Users\spagnuol\OneDrive - ISTAT\WP_13_Istat_Eurostat\Step2\Output\synthetic_data_datasetM10_TH20_RY_PR100_4CAT_MISS_X2_XGBoost_O0.csv


In [4]:
df_real.dtypes

municipality_residence    object
age                       object
civil_status              object
gender                    object
occupation                object
physical_activity         object
genetic_predisposition    object
diagnosis                 object
dtype: object

In [5]:
# Sintetizzatore XGBoost
# N.B.: Mettere add_noise a True per inserire rumore nelle numeriche 
#       altrimenti sarebbero identiche di default mentre con True è il 5% della std
synthesizer = XGBoostSynthesizer(random_state=42,add_noise=True)
df_synth = synthesizer.fit_sample(df_real, sample_size=10000,column_order=ordine_ottimale)
df_synth.to_csv(output_file_name,sep=',', index=False)


🔄 Sintesi incrementale: ['age', 'physical_activity', 'genetic_predisposition', 'municipality_residence', 'civil_status', 'gender', 'occupation', 'diagnosis']
🔄 Sintetizzando physical_activity
🔄 è categorica
🔄 Sintetizzando genetic_predisposition
🔄 è categorica
🔄 Sintetizzando municipality_residence
🔄 è categorica
🔄 Sintetizzando civil_status
🔄 è categorica
🔄 Sintetizzando gender
🔄 è categorica
🔄 Sintetizzando occupation
🔄 è categorica
🔄 Sintetizzando diagnosis
🔄 è categorica


In [6]:
df_synth.head()

,municipality_residence,age,civil_status,gender,occupation,physical_activity,genetic_predisposition,diagnosis
0,Ruffano,30,NeverMarried,Female,1,2,4,3
1,Faggiano,61,NeverMarried,Female,1,5,3,4
2,Lecce,53,NeverMarried,Female,0,3,2,3
3,Ruffano,49,Married,Female,1,3,4,2
4,Leverano,60,NeverMarried,Female,1,1,4,2


In [7]:
df_real.head()

,municipality_residence,age,civil_status,gender,occupation,physical_activity,genetic_predisposition,diagnosis
0,Corigliano d'Otranto,63,Married,Male,0,2,3,4
1,Faggiano,61,Married,Female,1,5,3,4
2,Taranto,53,NeverMarried,Male,0,3,3,3
3,Taranto,56,NeverMarried,Female,1,3,5,3
4,Lecce,20,Married,Female,0,1,4,1


In [8]:
df_synth.dtypes

municipality_residence    object
age                       object
civil_status              object
gender                    object
occupation                object
physical_activity         object
genetic_predisposition    object
diagnosis                 object
dtype: object

In [9]:
df_real.dtypes

municipality_residence    object
age                       object
civil_status              object
gender                    object
occupation                object
physical_activity         object
genetic_predisposition    object
diagnosis                 object
dtype: object

In [10]:
import io
from contextlib import redirect_stdout

#FUNZIONE MANUALE DI CONFRONTO TRA I 2 DF USO GLI INDICATORI PRINCIPALI
def confronto_completo_finale(df_real: pd.DataFrame, df_synth: pd.DataFrame):
    """🏆 CONFRONTO COMPLETO v4.0 - METRICHE SCIENTIFICHE COMPLETE"""
    
    import numpy as np
    from scipy import stats
    
    print("\n📚 METRICHE SCIENTIFICHE - LEGENDA")
    print("✅ NMAE  = Normalized Mean Absolute Error")
    print("✅ TVD   = Total Variation Distance") 
    print("✅ KS    = Kolmogorov-Smirnov (distrib. complete)")
    print("✅ KL    = Kullback-Leibler Divergence")
    print("✅ CORR  = Pearson Correlation preservata")
    print("✅ SDFS  = Synthetic Data Fidelity Score")
    print("✅ Privacy % = Exact row match avoidance")
    print("-" * 70)
    
    print("🏆 SYNTHETIC DATA QUALITY REPORT")
    print("=" * 90)
    
    # 1. RESET INDEX
    df_real_r = df_real.reset_index(drop=True)
    df_synth_r = df_synth.reset_index(drop=True)
    
    # 2. BASIC METRICS
    print(f"\n📊 DIMENSIONS")
    print(f"Real:     {df_real.shape}")
    print(f"Synthetic:{df_synth.shape}")
    print(f"Match:    {'✅' if df_real.shape == df_synth.shape else '❌'}")
    
    print(f"\n🔤 DATA TYPES")
    dtype_match = df_real.dtypes.equals(df_synth.dtypes)
    print(f"Exact match: {'✅' if dtype_match else '❌'}")
    
    # 3. NUMERIC COLUMNS - MULTIPLE METRICS
    numeric_cols = [col for col in df_real.select_dtypes([np.number]).columns 
                   if df_real[col].nunique() > 1]
    
    print(f"\n📈 NUMERIC COLUMNS ({len(numeric_cols)})")
    nmae_scores, ks_scores, kl_scores = {}, {}, {}
    
    for col in numeric_cols:
        r_data, s_data = df_real[col].dropna(), df_synth[col].dropna()
        
        # NMAE (medie)
        r_mean, s_mean = r_data.mean(), s_data.mean()
        r_std = r_data.std()
        nmae = abs(r_mean - s_mean) / max(r_std, 0.1)
        nmae_scores[col] = nmae
        
        # VERO KS test (distribuzioni complete)
        ks_stat, _ = stats.ks_2samp(r_data, s_data)
        ks_scores[col] = ks_stat
        
        # KL Divergence (distribuzioni)
        try:
            r_hist, s_hist = np.histogram(r_data, bins=20, density=True)[0], np.histogram(s_data, bins=20, density=True)[0]
            kl = stats.entropy(r_hist + 1e-10, s_hist + 1e-10)
            kl_scores[col] = kl
        except:
            kl_scores[col] = 0
            
        status = "🟢" if nmae < 0.1 and ks_stat < 0.1 else "🟡" if nmae < 0.3 else "🟠" if nmae < 0.5 else "🔴"
        print(f"  {col:15}: NMAE:{nmae:.3f} KS:{ks_stat:.3f} KL:{kl_scores[col]:.2f} | **{(1-nmae)*100:4.1f}%** {status}")
    
    nmae_mean = np.mean(list(nmae_scores.values()))
    ks_mean = np.mean(list(ks_scores.values()))
    
    # 4. CATEGORICAL COLUMNS
    print(f"\n🏷️ CATEGORICAL COLUMNS - TVD")
    cat_cols = df_real.select_dtypes(exclude=[np.number]).columns
    tvd_scores, kl_cat_scores = {}, {}
    
    for col in cat_cols:
        r_vc = df_real[col].value_counts(normalize=True)
        s_vc = df_synth[col].value_counts(normalize=True)
        tvd = abs(r_vc.reindex(s_vc.index, fill_value=0) - s_vc).sum() / 2
        tvd_scores[col] = tvd
        
        # KL per categoriche
        kl = stats.entropy(r_vc + 1e-10, s_vc.reindex(r_vc.index, fill_value=1e-10) + 1e-10)
        kl_cat_scores[col] = kl
        
        status = "🟢" if tvd < 0.05 else "🟡" if tvd < 0.15 else "🟠" if tvd < 0.3 else "🔴"
        print(f"  {col[:15]:15}: TVD:{tvd:.3f} KL:{kl:.2f} | **{(1-tvd)*100:4.1f}%** {status}")
    
    tvd_mean = np.mean(list(tvd_scores.values()))
    
    # 5. CORRELAZIONI (N.B. FACCIO LA PAIR COMBINATION TRA TUTTE LE NUMERICHE)
    print(f"\n📊 CORRELATION PRESERVATION")
    if len(numeric_cols) >= 2:
        corr_cols = numeric_cols[:4]
        r_corr_matrix = df_real[corr_cols].corr().values
        s_corr_matrix = df_synth[corr_cols].corr().values

        # Tutte le coppie 
        pairs = []
        for i in range(len(corr_cols)):
            for j in range(i+1, len(corr_cols)):
                r_corr = r_corr_matrix[i,j]
                s_corr = s_corr_matrix[i,j]
                diff = abs(r_corr - s_corr)
                pairs.append((corr_cols[i], corr_cols[j], r_corr, s_corr, diff))
                #pairs(colonna1, colonna2, correlazione_reale, correlazione_sintetica, differenza)
                print(f"  {corr_cols[i]}-{corr_cols[j]}: {r_corr:.3f}→{s_corr:.3f} (Δ:{diff:.3f})")

        corr_fidelity = 1 - np.mean([p[4] for p in pairs]) #p[4] corrisponde a diff 5 posizione
                                                           
        print(f"  📊 Correlation Fidelity: **{corr_fidelity*100:.1f}%** ({len(pairs)} pairs)")
    else:
        print("  Insufficient numeric columns")
        
    # 6. PRIVACY
    print(f"\n🔒 PRIVACY METRICS")
    print(f"Real duplicates:  {df_real.duplicated().sum()}")
    print(f"Synth duplicates: {df_synth.duplicated().sum()}")
    
    
    num_rows=len(df_real_r)
    identical = sum(1 for i in range(num_rows) 
                   if all(df_real_r.iloc[i][col] == df_synth_r.iloc[i][col] 
                         for col in df_real.columns if col in df_synth.columns))
    
    #bug fix identical
    real_tuples = set(tuple(row) for row in df_real.itertuples(index=False, name=None))
    synth_tuples = set(tuple(row) for row in df_synth.itertuples(index=False, name=None))

    # 2. Intersezione: record presenti in entrambi
    common_tuples = real_tuples.intersection(synth_tuples)

    # 3. Conta i record identici
    identical = len(common_tuples)    
    
    privacy_percent = (1 - identical/num_rows) * 100
    print(f"Identical rows: {identical}/{num_rows} | Privacy: **{privacy_percent:4.1f}%** ✅")
    
    # 7. COMPREHENSIVE SUMMARY
    print(f"\n📊 COMPREHENSIVE SUMMARY")
    print(f"  📈 NMAE:  {nmae_mean:.3f} → **{(1-nmae_mean)*100:4.1f}%**")
    print(f"  🎲 KS:    {ks_mean:.3f}")
    print(f"  🏷️ TVD:   {tvd_mean:.3f} → **{(1-tvd_mean)*100:4.1f}%**")
    print(f"  📊 Corr:  **{corr_preservation*100:.1f}%**" if 'corr_preservation' in locals() else "  📊 Corr:  N/A")
    print(f"  🔒 Privacy:**{privacy_percent:4.1f}%**")
    
    # 8. FINAL SDFS
    print("\n" + "="*90)
    #sdfs = 1 - (nmae_mean * 0.4 + tvd_mean * 0.4 + ks_mean * 0.2)
    #bug fix nan
    sdfs = 1 - ((nmae_mean if not np.isnan(nmae_mean) else 0) * 0.4 +
                (tvd_mean if not np.isnan(tvd_mean) else 0) * 0.4 +
                (ks_mean if not np.isnan(ks_mean) else 0) * 0.2)
    
    sdfs_percent = sdfs * 100
    
    print("🏆 SYNTHETIC DATA FIDELITY SCORE (SDFS)")
    print(f"🎯 FINAL QUALITY: **{sdfs_percent:5.1f}%**")
    
    if sdfs > 0.90:
        print("🟢 EXCELLENT (>90%) - Production Ready")
    elif sdfs > 0.80:
        print("🟡 GOOD (80-90%) - Excellent for testing")
    elif sdfs > 0.70:
        print("🟠 ADEQUATE (70-80%) - Usable")
    else:
        print("🔴 POOR (<70%) - Needs improvement")
    
    print(f"✅ XGBoostSynthesizer = {'🟢 PRODUCTION READY' if sdfs > 0.8 else '🔧 OPTIMIZE'}")
    
    return {
        'sdfs': sdfs,
        'sdfs_percent': sdfs_percent,
        'nmae_mean': nmae_mean,
        'ks_mean': ks_mean,
        'tvd_mean': tvd_mean,
        'privacy_percent': privacy_percent,
        'corr_preservation': corr_preservation if 'corr_preservation' in locals() else None,
        'nmae_scores': nmae_scores,
        'ks_scores': ks_scores,
        'tvd_scores': tvd_scores
    }


buffer = io.StringIO()

with redirect_stdout(buffer):
    risultati = confronto_completo_finale(df_real, df_synth)
    print(f"\n🎉 FINAL SDFS: {risultati['sdfs_percent']:.1f}%")

report_text = buffer.getvalue()

filename = f"Quality_Report_M10_TH20_R{R}_PR{PR}_4CAT_MISS_X2_{synthesizer_type}_O{ordine_scelto}.txt"
file_path = os.path.join(output_dir, filename)

with open(file_path, "w", encoding="utf-8") as f:
        f.write(report_text)
        
risultati = confronto_completo_finale(df_real, df_synth)
print(f"\n🎉 FINAL SDFS: {risultati['sdfs_percent']:.1f}%")

C:\Users\spagnuol\AppData\Local\anaconda3\envs\essnet_env\Lib\site-packages\numpy\_core\fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\spagnuol\AppData\Local\anaconda3\envs\essnet_env\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



📚 METRICHE SCIENTIFICHE - LEGENDA
✅ NMAE  = Normalized Mean Absolute Error
✅ TVD   = Total Variation Distance
✅ KS    = Kolmogorov-Smirnov (distrib. complete)
✅ KL    = Kullback-Leibler Divergence
✅ CORR  = Pearson Correlation preservata
✅ SDFS  = Synthetic Data Fidelity Score
✅ Privacy % = Exact row match avoidance
----------------------------------------------------------------------
🏆 SYNTHETIC DATA QUALITY REPORT

📊 DIMENSIONS
Real:     (10000, 8)
Synthetic:(10000, 8)
Match:    ✅

🔤 DATA TYPES
Exact match: ✅

📈 NUMERIC COLUMNS (0)

🏷️ CATEGORICAL COLUMNS - TVD
  municipality_re: TVD:0.042 KL:0.01 | **95.8%** 🟢
  age            : TVD:0.027 KL:0.00 | **97.3%** 🟢
  civil_status   : TVD:0.004 KL:0.00 | **99.6%** 🟢
  gender         : TVD:0.004 KL:0.00 | **99.6%** 🟢
  occupation     : TVD:0.004 KL:0.00 | **99.6%** 🟢
  physical_activi: TVD:0.010 KL:0.00 | **99.0%** 🟢
  genetic_predisp: TVD:0.011 KL:0.00 | **98.9%** 🟢
  diagnosis      : TVD:0.005 KL:0.00 | **99.5%** 🟢

📊 CORRELATION PRESE

C:\Users\spagnuol\AppData\Local\anaconda3\envs\essnet_env\Lib\site-packages\numpy\_core\fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\spagnuol\AppData\Local\anaconda3\envs\essnet_env\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Identical rows: 171/10000 | Privacy: **98.3%** ✅

📊 COMPREHENSIVE SUMMARY
  📈 NMAE:  nan → ** nan%**
  🎲 KS:    nan
  🏷️ TVD:   0.013 → **98.7%**
  📊 Corr:  N/A
  🔒 Privacy:**98.3%**

🏆 SYNTHETIC DATA FIDELITY SCORE (SDFS)
🎯 FINAL QUALITY: ** 99.5%**
🟢 EXCELLENT (>90%) - Production Ready
✅ XGBoostSynthesizer = 🟢 PRODUCTION READY

🎉 FINAL SDFS: 99.5%


In [11]:
### SINTESI DF_SYNTH DEI DUPLICATI

# Conta quante volte compare ciascun record
df_conteggi = (
    df_synth
    .groupby(list(df_synth.columns))
    .size()
    .reset_index(name='num_occorrenze')
)

# Totale record
tot_record = len(df_synth)

# Totale record unici (occurrence = 1)
tot_unici = (df_conteggi['num_occorrenze'] == 1).sum()

# Record con duplicati (occurrence > 1)
duplicati = df_conteggi.loc[df_conteggi['num_occorrenze'] > 1, 'num_occorrenze']
tot_duplicati_count = duplicati.count()       # quante righe con duplicati
tot_duplicati_sum = duplicati.sum()           # somma di tutte le occorrenze dei duplicati

# Stampa sintesi
print(f"Totale record: {tot_record}")
print(f"Totale record unici: {tot_unici}")
print(f"Record con duplicati (righe): {tot_duplicati_count}")
print(f"Record con duplicati (somma occorrenze): {tot_duplicati_sum}")

# DataFrame dei duplicati ordinati per occorrenze decrescenti
df_conteggi = (
    df_conteggi
    .query('num_occorrenze > 1')
    .sort_values('num_occorrenze', ascending=False)
    .reset_index(drop=True)
)

df_conteggi


Totale record: 10000
Totale record unici: 9885
Record con duplicati (righe): 57
Record con duplicati (somma occorrenze): 115


,municipality_residence,age,civil_status,gender,occupation,physical_activity,genetic_predisposition,diagnosis,num_occorrenze
0,Taranto,22,Married,Female,1,2,1,4,3
1,Brindisi,43,Married,Male,1,1,5,4,2
2,Brindisi,47,Married,Male,1,5,3,3,2
3,Brindisi,57,Married,Male,1,3,7,3,2
4,Brindisi,65,Married,Female,1,2,4,2,2
5,Cavallino,34,Married,Female,0,2,6,3,2
6,Cavallino,64,Married,Female,1,3,2,2,2
7,Corigliano d'Otranto,39,NeverMarried,Female,1,1,5,2,2
8,Alliste,56,NeverMarried,Male,1,2,4,4,2
9,Fasano,29,Married,Male,1,2,4,2,2


In [12]:
#FUNZIONE DI CONFRONTO TRA I 2 DF USO FRAMEWORK SDV
from sdv.metadata import SingleTableMetadata
from sdv.evaluation.single_table import run_diagnostic, evaluate_quality
import pandas as pd


def confronto_sdv_completo(df_real: pd.DataFrame, df_synth: pd.DataFrame):
    """🔥 SDV EVALUATION COMPLETA - FUNZIONANTE"""

    print("🔥 SDV EVALUATION COMPLETA")
    print("=" * 80)

    # 1. METADATA
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(df_real)
    print("✅ Metadata OK")

    # 2. DIAGNOSTIC REPORT
    print("\n📋 DIAGNOSTIC REPORT")
    diagnostic = run_diagnostic(
        real_data=df_real,
        synthetic_data=df_synth,
        metadata=metadata
    )
    diagnostic_score = diagnostic.get_score()    
    print(f"🎯 DIAGNOSTIC SCORE: {diagnostic_score:.1%}")

    # 3. QUALITY REPORT
    print("\n📊 QUALITY REPORT")
    quality_report = evaluate_quality(
        real_data=df_real,
        synthetic_data=df_synth,
        metadata=metadata
    )
    quality_score = quality_report.get_score()
    #print(f"🎯 QUALITY SCORE: {quality_score}")

    # 4. METRICHE DETTAGLIATE
    print("\n🏅 METRICHE PRINCIPALI")
    #print(f"  Column Shapes:      {quality_report.get_score('Column Shapes')}")
    #print(f"  Column Pair Trends: {quality_report.get_score('Column Pair Trends')}")
    #print(quality_report.get_details(property_name='Column Shapes'))
    # 5. VERDETTO FINALE
    print("\n" + "=" * 80)
    print("🏆 VERDETTO SYNTHESIZER")

    if quality_score >= 0.85:
        verdict = "🟢 ECCELLENTE - Production Ready!"
    elif quality_score >= 0.70:
        verdict = "🟡 BUONO - Usabile"
    else:
        verdict = "🟠 ADEGUATO - Da ottimizzare"

    print(verdict)
    print(f"📊 Quality:    {quality_score:.1%}")
    print(f"🔍 Diagnostic: {diagnostic_score:.1%}")

    return {
        "quality_score": quality_score,
        "diagnostic_score": diagnostic_score,
        "quality_report": quality_report
    }


# -------------------------------
# USO
# -------------------------------
print("\n2️⃣ SDV Evaluation")
risultati = confronto_sdv_completo(df_real, df_synth)
quality_report=risultati['quality_report']
diagnostic_score=risultati['diagnostic_score']



print("\n✅ COMPLETATO!")
print(f"🎯 Score finale: {risultati['quality_score']:.1%}")

display(quality_report.get_details(property_name='Column Shapes'))
display(quality_report.get_details(property_name='Column Pair Trends'))
display("Score",quality_report.get_score()*100)


2️⃣ SDV Evaluation
🔥 SDV EVALUATION COMPLETA
✅ Metadata OK

📋 DIAGNOSTIC REPORT
Generating report ...

(1/2) Evaluating Data Validity: |█████████████████████████████████████████████████████| 8/8 [00:00<00:00, 1071.24it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |█████████████████████████████████████████████████████| 1/1 [00:00<00:00, 723.90it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 100.0%

🎯 DIAGNOSTIC SCORE: 100.0%

📊 QUALITY REPORT
Generating report ...

(1/2) Evaluating Column Shapes: |██████████████████████████████████████████████████████| 8/8 [00:00<00:00, 442.43it/s]|
Column Shapes Score: 98.66%

(2/2) Evaluating Column Pair Trends: |███████████████████████████████████████████████| 28/28 [00:00<00:00, 406.61it/s]|
Column Pair Trends Score: nan%

Overall Score (Average): 98.66%


🏅 METRICHE PRINCIPALI

🏆 VERDETTO SYNTHESIZER
🟢 ECCELLENTE - Production Ready!
📊 Quality:    98.7%
🔍 Diagnostic: 100.0%

✅ COMPLETATO!
🎯 Score finale: 98.7%


,Column,Metric,Score
0,municipality_residence,TVComplement,0.9582
1,age,TVComplement,0.9729
2,civil_status,TVComplement,0.9956
3,gender,TVComplement,0.9961
4,occupation,TVComplement,0.9958
5,physical_activity,TVComplement,0.9903
6,genetic_predisposition,TVComplement,0.9887
7,diagnosis,TVComplement,0.9950


,Column 1,Column 2,Metric,Score,Real Correlation,Synthetic Correlation,Real Association,Meets Threshold?
0,municipality_residence,age,ContingencySimilarity,NaN,NaN,NaN,0.119229,False
1,municipality_residence,civil_status,ContingencySimilarity,NaN,NaN,NaN,0.115601,False
2,municipality_residence,gender,ContingencySimilarity,NaN,NaN,NaN,0.114795,False
3,municipality_residence,occupation,ContingencySimilarity,NaN,NaN,NaN,0.121223,False
4,municipality_residence,physical_activity,ContingencySimilarity,NaN,NaN,NaN,0.117145,False
5,municipality_residence,genetic_predisposition,ContingencySimilarity,NaN,NaN,NaN,0.114743,False
6,municipality_residence,diagnosis,ContingencySimilarity,NaN,NaN,NaN,0.119183,False
7,age,civil_status,ContingencySimilarity,NaN,NaN,NaN,0.064544,False
8,age,gender,ContingencySimilarity,NaN,NaN,NaN,0.069795,False
9,age,occupation,ContingencySimilarity,NaN,NaN,NaN,0.059591,False


'Score'

np.float64(98.6575)

In [13]:
#ENTROPIA DEL DF_REAL
import pandas as pd
import numpy as np

df = df_real

# Funzione entropia discreta
def entropy_discrete(series):
    probs = series.dropna().value_counts(normalize=True)
    return -np.sum(probs * np.log2(probs))

# Funzione entropia per variabili continue
def entropy_continuous(series, bins=10):
    counts, _ = np.histogram(series.dropna(), bins=bins)
    probs = counts / counts.sum()
    probs = probs[probs > 0]  # evita log(0)
    return -np.sum(probs * np.log2(probs))

# Funzione per l’interpretazione automatica
def interpret_feature(row):
    h_norm = row['Entropia_Normalizzata']
    perc_unici = row['%_Unici']
    
    if h_norm >= 0.95:
        if perc_unici > 50:
            return 'Probabile ID / Rumore'
        else:
            return 'Feature ben bilanciata / Informativa'
    elif 0.6 <= h_norm < 0.95:
        return 'Feature con buona variabilità'
    elif 0.3 <= h_norm < 0.6:
        return 'Variabile moderatamente informativa'
    elif 0.05 <= h_norm < 0.3:
        return 'Poco informativa / sbilanciata'
    else:
        return 'Quasi costante / da scartare'

# Calcoli entropia
entropia = []
entropia_max = []
entropia_norm = []

for col in df.columns:
    s = df[col].dropna()
    H = entropy_discrete(s)
    entropia.append(H)
    k = s.nunique()
    H_max = np.log2(k) if k > 1 else 0
    entropia_max.append(H_max)
    entropia_norm.append(H / H_max if H_max > 0 else 0)

# Creazione summary
summary = pd.DataFrame({
    'Colonna': df.columns,
    'Tipo_Dati': df.dtypes.values,
    'Valori_Totali': len(df),
    'Valori_Unici': df.nunique().values,
    '%_Unici': (df.nunique() / len(df) * 100).round(2),
    '%_Nulli': (df.isnull().sum() / len(df) * 100).round(1),
    'Entropia': np.round(entropia, 3),
    'Entropia_Max': np.round(entropia_max, 3),
    'Entropia_Normalizzata': np.round(entropia_norm, 3)
})

# Aggiunta colonna interpretazione
summary['Interpretazione_Automatica'] = summary.apply(interpret_feature, axis=1)
summary.reset_index(drop=True, inplace=True)
summary


,Colonna,Tipo_Dati,Valori_Totali,Valori_Unici,%_Unici,%_Nulli,Entropia,Entropia_Max,Entropia_Normalizzata,Interpretazione_Automatica
0,municipality_residence,object,10000,145,1.45,0.0,6.317,7.180,0.880,Feature con buona variabilità
1,age,object,10000,48,0.48,0.0,5.554,5.585,0.995,Feature ben bilanciata / Informativa
2,civil_status,object,10000,5,0.05,0.0,1.315,2.322,0.566,Variabile moderatamente informativa
3,gender,object,10000,2,0.02,0.0,1.000,1.000,1.000,Feature ben bilanciata / Informativa
4,occupation,object,10000,2,0.02,0.0,0.938,1.000,0.938,Feature con buona variabilità
5,physical_activity,object,10000,6,0.06,0.0,2.113,2.585,0.818,Feature con buona variabilità
6,genetic_predisposition,object,10000,9,0.09,0.0,2.756,3.170,0.869,Feature con buona variabilità
7,diagnosis,object,10000,4,0.04,0.0,2.000,2.000,1.000,Feature ben bilanciata / Informativa


In [14]:
#SAVE FILE EXCEL PER RICCIATO
#summary.to_excel(f'{output_dir}/Ricciato_summaryEntropia_data_datasetM10_TH20_R{R}_PR{PR}_4CAT_MISS_X2.xlsx', index=False)

In [15]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.preprocessing import LabelEncoder

# ---- INPUT ----
df = df_real.copy()
target_col = 'diagnosis'  # sostituisci con il nome reale del target
task_type = 'classification'  # 'classification' x target categoriali
task_type = 'regression'  # 'regression' x target num continui

# ---- Funzioni entropia ----
def entropy_discrete(series):
    probs = series.dropna().value_counts(normalize=True)
    return -np.sum(probs * np.log2(probs)) if len(probs) > 0 else 0

def entropy_continuous(series, bins=10):
    counts, _ = np.histogram(series.dropna(), bins=bins)
    probs = counts / counts.sum()
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

# ---- Funzione interpretazione automatica ----
def interpret_feature(row):
    h_norm = row['Entropia_Normalizzata']
    perc_unici = row['%_Unici']
    
    if h_norm >= 0.95:
        if perc_unici > 50:
            return 'Probabile ID / Rumore'
        else:
            return 'Feature ben bilanciata / Informativa'
    elif 0.6 <= h_norm < 0.95:
        return 'Feature con buona variabilità'
    elif 0.3 <= h_norm < 0.6:
        return 'Variabile moderatamente informativa'
    elif 0.05 <= h_norm < 0.3:
        return 'Poco informativa / sbilanciata'
    else:
        return 'Quasi costante / da scartare'

# ---- Calcolo entropia ----
entropia = []
entropia_max = []
entropia_norm = []

for col in df.columns:
    s = df[col].dropna()
    H = entropy_discrete(s)
    entropia.append(H)
    k = s.nunique()
    H_max = np.log2(k) if k > 1 else 0
    entropia_max.append(H_max)
    entropia_norm.append(H / H_max if H_max > 0 else 0)

# ---- Creazione summary base ----
summary = pd.DataFrame({
    'Colonna': df.columns,
    'Tipo_Dati': df.dtypes.values,
    'Valori_Totali': len(df),
    'Valori_Unici': df.nunique().values,
    '%_Unici': (df.nunique() / len(df) * 100).round(2),
    'Valori_Nulli': df.isnull().sum().values,
    '%_Nulli': (df.isnull().sum() / len(df) * 100).round(1),
    'Entropia': np.round(entropia, 3),
    'Entropia_Max': np.round(entropia_max, 3),
    'Entropia_Normalizzata': np.round(entropia_norm, 3)
})

# ---- Codifica categorical per MI ----
X = df.drop(columns=[target_col]).copy()
y = df[target_col]
le_dict = {}
for col in X.select_dtypes(include='object').columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    le_dict[col] = le  # opzionale, per decodifica futura

# ---- Mutual Information ----
if task_type == 'classification':
    mi = mutual_info_classif(X, y, discrete_features='auto', random_state=42)
else:
    mi = mutual_info_regression(X, y, discrete_features='auto', random_state=42)

mi_df = pd.DataFrame({'Colonna': X.columns, 'MI': mi})

# ---- Merge MI con summary ----
summary = summary.merge(mi_df, on='Colonna', how='left')

# ---- Interpretazione automatica ----
summary['Interpretazione_Automatica'] = summary.apply(interpret_feature, axis=1)

# ---- Ordinamento summary per MI decrescente ----
summary = summary.sort_values('MI', ascending=False).reset_index(drop=True)

# ---- Lista feature ordinate per MI (pronta per XGBoost) ----
feature_order = summary['Colonna'].tolist()
#feature_order = feature_order 

# ---- Output ----
print("Feature order (dal più informativo):")
print(feature_order)

#summary.to_csv(f'{output_dir}/Ricciato_summary_real_data_datasetM10_TH20_R{R}_PR{PR}_4CAT_MISS_X2.csv',sep=',', index=False)

summary


Feature order (dal più informativo):
['genetic_predisposition', 'physical_activity', 'age', 'occupation', 'municipality_residence', 'civil_status', 'gender', 'diagnosis']


,Colonna,Tipo_Dati,Valori_Totali,Valori_Unici,%_Unici,Valori_Nulli,%_Nulli,Entropia,Entropia_Max,Entropia_Normalizzata,MI,Interpretazione_Automatica
0,genetic_predisposition,object,10000,9,0.09,0,0.0,2.756,3.170,0.869,0.041237,Feature con buona variabilità
1,physical_activity,object,10000,6,0.06,0,0.0,2.113,2.585,0.818,0.031825,Feature con buona variabilità
2,age,object,10000,48,0.48,0,0.0,5.554,5.585,0.995,0.009866,Feature ben bilanciata / Informativa
3,occupation,object,10000,2,0.02,0,0.0,0.938,1.000,0.938,0.003528,Feature con buona variabilità
4,municipality_residence,object,10000,145,1.45,0,0.0,6.317,7.180,0.880,0.000721,Feature con buona variabilità
5,civil_status,object,10000,5,0.05,0,0.0,1.315,2.322,0.566,0.000000,Variabile moderatamente informativa
6,gender,object,10000,2,0.02,0,0.0,1.000,1.000,1.000,0.000000,Feature ben bilanciata / Informativa
7,diagnosis,object,10000,4,0.04,0,0.0,2.000,2.000,1.000,NaN,Feature ben bilanciata / Informativa
